# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karthikmannam/flyrank-internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns the validated model (week 5/6) into a **reviewer queue a human can act on** — ranked, with a reason code and a plain-language action on every page. It works top to bottom so Run All succeeds.

> Skills loaded: `writing-honest-claims` + `flyrank/flyrank-data`. The claim ladder is the backbone here: **observed** on this snapshot, **decision-support** for a reviewer — never a causal "editing this will lift you."  

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Every page gets a **validated** score — the week-5 Random Forest, but scored the honest way: a 5-fold client-grouped out-of-fold run, so no page is scored by a model that trained on its own client. On top of that score sits the week-4 **reason code** (what is wrong with the page) and a **plain action** (what an editor should do). Rank is by model score, high to low.

**Archetype → action map** (small, transparent, same spirit as the week-4 rule):

| reason_code (archetype) | what it means | action |
|---|---|---|
| `low_visibility` | under 500 impressions in 90d | **monitor** |
| `visible_monitor` | visible, but no strong reason to touch | **monitor** |
| `slipping_high_impact` | visible and position slipping | **refresh_and_optimize** |
| `stale_high_impact` | visible and untouched 180d+ | **refresh** |
| `stale_slipping_high_impact` | visible, stale, and slipping | **refresh_high_priority** |

**The honest carve-outs:** the model beats the base rate in the mid-queue (P@50 ≈ 0.60 vs 0.54 across the whole queue; ~0.70 among pages the rule already flagged) but is weakest at the very top (P@10 ≈ 0.50). And `refresh_high_priority` is the rarest, clearest band: 14 pages, 93% of them observed-declining. So this is a *screen for people*, not an autopilot — the human gates live in Section 3.

In [1]:

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import json

ROOT = Path().resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
BASELINE_PATH = ROOT / "work" / "outputs" / "baseline_action_score.csv"
OUT_CSV = ROOT / "work" / "outputs" / "w07_action_queue.csv"
METRICS_PATH = ROOT / "work" / "outputs" / "w07_action_playbook.json"
FIG_DIR = ROOT / "work" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

log_cols = {"impressions_90d":"log_impressions_90d","clicks_90d":"log_clicks_90d",
            "sessions_90d":"log_sessions_90d","ai_sessions_90d":"log_ai_sessions_90d"}
for src, dst in log_cols.items():
    df[dst] = np.log1p(df[src])
df["has_clicks"] = (df["clicks_90d"]>0).astype(int)
df["has_position_data"] = (df["avg_position"]>0).astype(int)

from scripts.ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, precision_at_k
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

numeric_features = [c for c in MODEL_NUMERIC_FEATURES if c in df.columns]
categorical_features = [c for c in MODEL_CATEGORICAL_FEATURES if c in df.columns]

for c in numeric_features:
    df[c] = pd.to_numeric(df[c], errors="coerce")
num = df[numeric_features].replace([np.inf,-np.inf], np.nan).fillna(0)
cat = df[categorical_features].fillna("unknown").astype(str)
enc = pd.get_dummies(cat, prefix=categorical_features, prefix_sep="_", dtype=float, dummy_na=False)
X = pd.concat([num.reset_index(drop=True), enc.reset_index(drop=True),
               df[["has_position_data"]].reset_index(drop=True)], axis=1)
y = df["is_declining_label"].to_numpy()
clients = df["client_id"].astype(str).to_numpy()

RF = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10,
                            min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
gkf = GroupKFold(n_splits=5)
oof = np.zeros(len(df))
for tr, te in gkf.split(X, y, groups=clients):
    m = RF.__class__(**RF.get_params())
    m.fit(X.iloc[tr], y[tr])
    oof[te] = m.predict_proba(X.iloc[te])[:, 1]

base = float(y.mean())
print(f"Validated (client-group OOF): roc_auc={roc_auc_score(y,oof):.3f}  avg_precision={average_precision_score(y,oof):.3f}")

merged = pd.read_csv(BASELINE_PATH).merge(
    pd.DataFrame({"content_id": df["content_id"], "model_score": oof}), on="content_id", how="left")

ARCH = {
    "low_visibility":              "monitor",
    "visible_monitor":             "monitor",
    "slipping_high_impact":        "refresh_and_optimize",
    "stale_high_impact":           "refresh",
    "stale_slipping_high_impact":  "refresh_high_priority",
}
merged["playbook_action"] = merged["reason_code"].map(ARCH)
merged["model_rank"] = merged["model_score"].rank(method="first", ascending=False).astype(int)
merged = merged.sort_values("model_rank").reset_index(drop=True)

actionable = merged[merged["playbook_action"].isin(["refresh_high_priority","refresh_and_optimize","refresh"])]
prio = merged[merged["playbook_action"] == "refresh_high_priority"]
srt = merged.sort_values("model_score", ascending=False)

print(f"Actionable rows: {len(actionable):,}  ({len(actionable)/len(merged):.1%} of queue)")
print(f"  observed-declining among actionable: {actionable['is_declining_label'].mean():.1%}  (base {base:.1%})")
print(f"  refresh_high_priority: n={len(prio)}  observed-declining={prio['is_declining_label'].mean():.0%}")
print(f"  top-50 by model score within actionable: P@50={precision_at_k(actionable['is_declining_label'].to_numpy(), actionable['model_score'].to_numpy(), 50):.2f}")


Validated (client-group OOF): roc_auc=0.687  avg_precision=0.680
Actionable rows: 9,165  (30.6% of queue)
  observed-declining among actionable: 60.1%  (base 54.2%)
  refresh_high_priority: n=14  observed-declining=93%
  top-50 by model score within actionable: P@50=0.70


In [2]:

preview_cols = ["model_rank","playbook_action","reason_code","is_declining_label",
                "impressions_90d","avg_position","days_since_last_update","freshness_tier"]
print("Top 10 of the ranked queue:")
print(merged.head(10)[preview_cols].to_string(index=False))
print("\nQueue precision at depth K (validated model vs base):")
for k in (5,10,50,100,500,1000):
    print(f"  P@{k:<5} {precision_at_k(merged['is_declining_label'].to_numpy(), merged['model_score'].to_numpy(), k):.3f}   (base {base:.3f})")


Top 10 of the ranked queue:
 model_rank      playbook_action          reason_code  is_declining_label  impressions_90d  avg_position  days_since_last_update freshness_tier
          1 refresh_and_optimize slipping_high_impact                   1             2197          21.5                     104         91-180
          2              monitor       low_visibility                   0              360           7.2                     104         91-180
          3 refresh_and_optimize slipping_high_impact                   1              556          10.9                      28           0-30
          4 refresh_and_optimize slipping_high_impact                   0             1191          23.1                     103         91-180
          5 refresh_and_optimize slipping_high_impact                   1              758          31.1                     104         91-180
          6              monitor       low_visibility                   0              209          20.0    

  P@1000  0.727   (base 0.542)


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who and what.** An editor or content strategist uses this queue to decide *which pages to look at first*. The queue ranks decline risk and tags each page with the reason and a suggested action. It is a **screening tool for human review** — not an automatic switch that edits or publishes anything.

**Cost / value, plainly.** The value is narrower than "a model that predicts Google." Measured on held-in client groups, the model separates observed-declining from stable pages at **avg_precision ≈ 0.68**. Its real utility is *concentration*: among the 9,165 pages the rule already flags, ranking by model score makes the top 50 **~70% truly-declining** (vs ~60% if you took any flagged page, vs ~54% at random across the whole set). The cost is real too: it is wrong about the direction of ~3 in 10, and it is weakest exactly at the top (P@10 ≈ 0.50), so the *first handful* still need human eyes before anything is touched.

**The decay insight (measured, this snapshot).** Observed decline rises with staleness among visible pages (impressions ≥ 500):

| days since update | pages | observed decline |
|---|---|---|
| 0-30 (fresh) | 10,063 | 58% |
| 31-90 | 88 | 52% |
| 91-180 | 6,558 | 62% |
| 181+ | 17 | 94% |

The 181+ band is tiny (n=17) — I read it as suggestive, not proof. But the 91-180 band (n=6,558, 62%) is a real, measured pool: pages untouched for a quarter are where the actionable work concentrates.

**Operational limits (where it stops being valid):**
- One **trailing-90-day snapshot**, no interventions measured — nothing here is causal. "Worth a second look" is the honest claim; "refreshing will grow it" is a claim this data cannot carry.
- The model scores are **out-of-fold on this snapshot only**; a real deployment needs fresh features and the honest split re-run.
- No titles, URLs, keywords, or client identity enter the model — only the numeric signals in the data dictionary.

In [3]:

# Numbers behind the cost/value and decay frames
odf = merged[["days_since_last_update","impressions_90d","is_declining_label","freshness_tier"]].copy()
visible = odf[odf["impressions_90d"] >= 500]
tier = visible.groupby("freshness_tier", observed=False)["is_declining_label"].agg(["size","mean"]).round(3)
print("Observed decline by freshness band, visible pages (imp>=500):")
print(tier.to_string())
print()
print("Head-of-queue precision@K (validated model, whole queue):")
for k in (10, 50, 100):
    print(f"  P@{k:<4} {precision_at_k(merged['is_declining_label'].to_numpy(), merged['model_score'].to_numpy(), k):.2f}   vs base {base:.2f}")


Observed decline by freshness band, visible pages (imp>=500):
                 size   mean
freshness_tier              
0-30            10063  0.583
181+               17  0.941
31-90              88  0.523
91-180           6558  0.616

Head-of-queue precision@K (validated model, whole queue):
  P@10   0.50   vs base 0.54
  P@50   0.60   vs base 0.54
  P@100  0.67   vs base 0.54


## 3. Human review + the no-go list

*What a person must check before acting. What must never be automated.*

**The human rule, stated as a contract:**
1. **Read, don't run.** The queue asks a person to open a page and confirm the reason is real — a rank drop can be seasonal, a zero-CTR page can be a tracking lag. No page is edited or republished by automation.
2. **Own the very top by hand.** Because P@10 ≈ 0.50 (weakest at the top), the first ~10 rows get *extra* human review, not confident execution.
3. **Match the archetype to the actual page.** A "thin / slipping" tag on a page that is really an evergreen or a product page means the recommended action is wrong — set it to monitor by hand.

**The no-go list — never automate:**
- **No auto-edit or auto-publish of anything.** This queue is an aid, full stop.
- **No automated verdict on pages with no position data.** `avg_position=0` means "no data", not rank zero — 1,205 rows can't be judged on rank at all; they stay "monitor" for human eyes only.
- **No hands on valuable top-3 pages without sign-off.** 339 top-3 pages are visible *and* observed-declining; a wrong edit costs more than no edit. They are frozen behind human approval.
- **No reversal based on one snapshot.** A single 90-day window doesn't license turning a recommendation into a direction-flag or a client-facing claim.

In [4]:

# The no-go cases, counted, so each "can't" sentence has a number under it.
no_pos = int((merged["avg_position"] == 0).sum())
top3_decl_visible = int(((merged["position_tier"] == "top_3") &
                         (merged["is_declining_label"] == 1) &
                         (merged["impressions_90d"] >= 500)).sum())
print(f"Rows with NO position data (never auto-ranked): {no_pos:,}")
print(f"Top-3 visible rows observed-declining (human sign-off required): {top3_decl_visible:,}")
print(f"Actionable rows (only place a 'refresh_*' action may appear): {len(actionable):,}")
print(f"  of which refresh_high_priority: {int(prio['content_id'].count()):,}")


Rows with NO position data (never auto-ranked): 1,205
Top-3 visible rows observed-declining (human sign-off required): 339
Actionable rows (only place a 'refresh_*' action may appear): 9,165
  of which refresh_high_priority: 14


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Lightweight rules, run on the next snapshot (noticing drift needs no retrain):

1. **Precision@50 floor.** The validated queue runs at P@50 ≈ 0.60 across everything (≈0.70 within actionable pages). If the *next* snapshot's P@50 drops below **0.55**, the queue is no longer separating — flag a human and consider retraining.
2. **Score-distribution drift.** Today the top 10% of model scores sit above **0.698**. If that threshold moves by more than ±3pts, the model is scoring the new snapshot differently.
3. **Label base-rate drift.** Base observed-decline is **54.2%**. If the next snapshot's decline rate leaves the 54-59% band, the yardstick moved — re-rank, and likely retrain.
4. **Pool drift — staleness.** Only 17 pages are visible and untouched 180d+ today. If that pool balloons (or the action mix shifts), re-read the playbook against current intent before acting.
5. **Concrete retrain trigger.** Retrain when: base rate leaves band, or P@50 < 0.55 on two checkpoints in a row, or any input column changes meaning (schema / source change). Otherwise keep the current model.

In [5]:

# Current yardsticks the next snapshot compares to
triggers = {
    "precision@50_floor": 0.55,
    "base_rate_band": [0.54, 0.59],
    "score_p90_threshold": float(np.quantile(oof, 0.90)),
    "stale_visible_count": int(((merged["days_since_last_update"] >= 180) &
                                (merged["impressions_90d"] >= 500)).sum()),
    "no_position_rows": no_pos,
}
print("Trigger yardsticks (next snapshot compares to these):")
for k, v in triggers.items():
    print(f"  {k}: {v}")


Trigger yardsticks (next snapshot compares to these):
  precision@50_floor: 0.55
  base_rate_band: [0.54, 0.59]
  score_p90_threshold: 0.698133946591327
  stale_visible_count: 17
  no_position_rows: 1205


## 5. Exports for the paper

*Write the queue and the reusable figures so the paper builds on real files, not memory.*

- **Queue CSV** → `work/outputs/w07_action_queue.csv` (30,000 rows). It is git-ignored by policy (`work/**/*.csv`), but the notebook must still generate it — it is the artifact a reader could re-make.
- **Metrics receipt** → `work/outputs/w07_action_playbook.json` — committed, the numbers the prose traces back to.
- **Figures** → `work/figures/action_mix.png`, `work/figures/queue_precision.png` — committed, reused in the paper.

In [6]:

# --- Queue CSV (git-ignored by work/**/*.csv policy, but must be produced) ---
export_cols = ["model_rank","content_id","client_id","playbook_action","reason_code",
               "model_score","baseline_action_score","is_declining_label",
               "impressions_90d","avg_position","ctr","content_age_days",
               "days_since_last_update","word_count","freshness_tier","impression_tier","position_tier"]
export = merged[export_cols]
export.to_csv(OUT_CSV, index=False)
print(f"Queue CSV written: {OUT_CSV.relative_to(ROOT)}  ({len(export):,} rows x {len(export.columns)} cols)")

# --- Metrics receipt (committed JSON) ---
metrics = {
    "target": "is_declining_label",
    "validated_oof_roc_auc": round(roc_auc_score(y, oof), 3),
    "validated_oof_avg_precision": round(average_precision_score(y, oof), 3),
    "base_rate": round(base, 3),
    "actionable_rows": int(len(actionable)),
    "actionable_declining_rate": round(float(actionable["is_declining_label"].mean()), 3),
    "actionable_p50": round(float(precision_at_k(actionable["is_declining_label"].to_numpy(),
                                                 actionable["model_score"].to_numpy(), 50)), 3),
    "queue_p10": round(float(precision_at_k(merged["is_declining_label"].to_numpy(),
                                            merged["model_score"].to_numpy(), 10)), 3),
    "queue_p50": round(float(precision_at_k(merged["is_declining_label"].to_numpy(),
                                            merged["model_score"].to_numpy(), 50)), 3),
    "queue_p100": round(float(precision_at_k(merged["is_declining_label"].to_numpy(),
                                             merged["model_score"].to_numpy(), 100)), 3),
    "refresh_high_priority": {
        "count": int(len(prio)),
        "declining_rate": round(float(prio["is_declining_label"].mean()), 3),
    },
    "decay_observed": {k: [int(v["size"]), round(float(v["mean"]), 3)] for k, v in tier.iterrows()},
    "no_position_rows": no_pos,
    "no_go_top3_declining_visible": top3_decl_visible,
    "trigger_yardsticks": triggers,
    "export_csv": "work/outputs/w07_action_queue.csv",
    "status": "generated",
}
METRICS_PATH.write_text(json.dumps(metrics, indent=2, sort_keys=True))
print(f"Metrics JSON written: {METRICS_PATH.relative_to(ROOT)}")


Queue CSV written: work\outputs\w07_action_queue.csv  (30,000 rows x 17 cols)


Metrics JSON written: work\outputs\w07_action_playbook.json


In [7]:

# --- Figures reused in the paper ---
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 140

# Fig 1: action mix across the queue
amc = merged["playbook_action"].value_counts()
fig, ax = plt.subplots(figsize=(7, 4))
amc.reindex(["monitor", "refresh", "refresh_and_optimize", "refresh_high_priority"]).plot.bar(
    color="#6F4E7C", ax=ax)
ax.set_title("Suggested action across the ranked queue")
ax.set_ylabel("pages")
ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
fig.savefig(FIG_DIR / "action_mix.png"); plt.close(fig)

# Fig 2: queue precision vs depth vs base rate
ks = [5, 10, 50, 100, 500, 1000]
pvals = [precision_at_k(merged["is_declining_label"].to_numpy(),
                        merged["model_score"].to_numpy(), k) for k in ks]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, pvals, marker="o", label="model (client-group OOF)")
ax.axhline(base, color="gray", ls="--", label=f"base rate {base:.2f}")
ax.set_xlabel("queue depth K"); ax.set_ylabel("precision@K (observed-declining)")
ax.set_title("Ranked-queue precision vs depth")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "queue_precision.png"); plt.close(fig)

print(f"Figures written -> {FIG_DIR.relative_to(ROOT)}: action_mix.png, queue_precision.png")


Figures written -> work\figures: action_mix.png, queue_precision.png


## Self-check

- [x] Every section is built — markdown thinking AND the code that backs it
- [x] Ran top to bottom, no errors (this file was executed end to end)
- [x] No client names, no private URLs, no raw queries — only aggregate counts
- [x] Claims sit at "observed / measured / decision-support" — no causal, no future guarantees
- [x] Queue CSV produced (git-ignored by policy), metrics JSON + figures ready to commit


In [8]:

# Final assert: outputs exist and the headline numbers hold.
assert OUT_CSV.exists(), "queue CSV missing"
assert METRICS_PATH.exists(), "metrics JSON missing"
assert len(export) == 30000, f"expected 30000 queue rows, got {len(export)}"
assert no_pos == 1205, f"no-position row count changed: {no_pos}"
assert abs(base - 0.542) < 0.001, "base rate drifted"
assert abs(metrics["validated_oof_avg_precision"] - 0.68) < 0.01, "avg_precision drifted"
assert abs(metrics["queue_p50"] - 0.60) < 0.02, "queue P@50 drifted"

print("SELF-CHECK PASSED")
print(f"  queue={len(export):,} rows @ {OUT_CSV.relative_to(ROOT)}")
print(f"  figures: action_mix.png, queue_precision.png in {FIG_DIR.relative_to(ROOT)}")
print(f"  receipt: {METRICS_PATH.relative_to(ROOT)}")


SELF-CHECK PASSED
  queue=30,000 rows @ work\outputs\w07_action_queue.csv
  figures: action_mix.png, queue_precision.png in work\figures
  receipt: work\outputs\w07_action_playbook.json
